# CSV Preparation

In [ ]:
# dataset_csv = "dataset.csv"

# # Read each line from the CSV file and store it in a list
# number_of_videos = 0
# number_of_class = 0
# with open(dataset_csv, "r") as f:
#     for line in f:
#         number_of_vids = line.strip().split()[-1]
#         number_of_videos += int(number_of_vids)
#         number_of_class += 1

# print(f"Total number of videos: {number_of_videos}")
# print(f"Total number of classes: {number_of_class}")
# print(
#     f"Average number of videos per class: {number_of_videos / number_of_class}"
# )

Total number of videos: 220847
Total number of classes: 174
Average number of videos per class: 1269.2356321839081


In [20]:
import os
import random


DATASET_NAME = "vjepa-prob-dataset"
SUPPORT_EXTS = (".mp4", ".webp")
train_csv = os.path.join("datasets", DATASET_NAME, "train.csv")
val_csv = os.path.join("datasets", DATASET_NAME, "val.csv")
train_dataset_dir = os.path.join("datasets", DATASET_NAME, "train")
val_dataset_dir = os.path.join("datasets", DATASET_NAME, "val")

# Class names
class_names = [
    i
    for i in os.listdir(train_dataset_dir)
    if os.path.isdir(os.path.join(train_dataset_dir, i))
]

train_list = []
val_list = []
for classname in class_names:
    classname = classname.lower()
    if (
        ("non_" in classname)
        or ("non-" in classname)
        or (classname == "normal")
    ):
        class_index = 0
    else:
        class_index = 1

    train_class_dir = os.path.join(train_dataset_dir, classname)
    val_class_dir = os.path.join(val_dataset_dir, classname)

    # Training
    for filename in os.listdir(train_class_dir):
        if not filename.endswith(SUPPORT_EXTS):
            continue
        filepath = os.path.join(train_class_dir, filename)
        train_list.append(f"{filepath} {class_index}")

    # Validation
    for filename in os.listdir(val_class_dir):
        if not filename.endswith(SUPPORT_EXTS):
            continue
        filepath = os.path.join(val_class_dir, filename)
        val_list.append(f"{filepath} {class_index}")

random.shuffle(train_list)
random.shuffle(val_list)

with open(train_csv, "w") as f:
    for item in train_list:
        f.write(item + "\n")

with open(val_csv, "w") as f:
    for item in val_list:
        f.write(item + "\n")

In [21]:
import os

DATASET = os.path.join("datasets", DATASET_NAME)


def get_video_paths(dir: str) -> list:
    video_paths = []
    for root, _, files in os.walk(dir):
        for file in files:
            if file.endswith((".mp4", ".avi", ".mov", ".webm")):
                video_paths.append(os.path.join(root, file))

    return video_paths


video_paths = get_video_paths(DATASET)
print(f"Found {len(video_paths)} video files in {DATASET}")

Found 1551 video files in datasets/vjepa-prob-dataset


In [23]:
import cv2


average_duration: float = 0
average_frames: float = 0
average_fps: float = 0
for video_path in video_paths:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Failed to open video: {video_path}")
        continue

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = frame_count / fps if fps > 0 else 0

    average_duration += duration
    average_frames += frame_count
    average_fps += fps

average_duration /= len(video_paths)
average_frames /= len(video_paths)
average_fps /= len(video_paths)

print(f"Average duration: {average_duration}")
print(f"Average frames: {average_frames}")
print(f"Average FPS: {average_fps}")
print(f"Prefer num_segments: {int(average_frames / 64)}")

Average duration: 16.972896527914227
Average frames: 437.47130883301094
Average FPS: 26.352450824481743
Prefer num_segments: 6


In [ ]:
import os
import shutil
import json
import cv2
import random

DATASET_DIR = "/Volumes/TeeExt/"
TRAIN_RATIO = 0.8


def get_video_paths(dir: str) -> list:
    videos = []
    for root, _, files in os.walk(dir):
        for file in files:
            if file.lower().endswith((".mp4", ".avi")):
                videos.append(os.path.join(root, file))
    return videos


def trim_video(
    input_path: str,
    output_path: str,
    start_frame: int,
    end_frame: int,
) -> None:
    cap = cv2.VideoCapture(input_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter.fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    for frame_num in range(start_frame, end_frame + 1):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)

    cap.release()
    out.release()


violence_dir = os.path.join(DATASET_DIR, "violence")
non_violence_dir = os.path.join(DATASET_DIR, "non-violence")

trainable_violences = [
    i for i in get_video_paths(violence_dir) if "gtav" not in i
]
trainable_non_violences = [
    i for i in get_video_paths(non_violence_dir) if "gtav" not in i
]

trainable_dataset = {}
for i, trainables in enumerate([trainable_violences, trainable_non_violences]):
    dataset_info = {}
    for trainable in trainables:
        video_name, ext = os.path.splitext(os.path.basename(trainable))
        dataset_name = video_name.split("_video_")[0]
        if dataset_name not in dataset_info:
            dataset_info[dataset_name] = [trainable]
        else:
            dataset_info[dataset_name].append(trainable)

    if i == 0:
        trainable_dataset["violence"] = dataset_info
    else:
        trainable_dataset["non-violence"] = dataset_info

trains = []
vals = []
for dirname, data in trainable_dataset.items():
    for dataset_name, videos_list in data.items():
        random.shuffle(videos_list)
        num_train = len(videos_list) * TRAIN_RATIO
        train_list = videos_list[:num_train]
        val_list = videos_list[num_train + 1 :]

In [3]:
# UCF-Crime
import json
import os

DATASET_DIR = "/Volumes/TeeExt/ucf-crime"
VIDEO_DIR = "/Volumes/TeeExt/ucf-crime/UCF_Crimes/UCF_Crimes/Videos/Shooting"


def get_video_paths(dir: str) -> list:
    videos = []
    for root, _, files in os.walk(dir):
        for file in files:
            if file.lower().endswith((".mp4", ".avi")):
                videos.append(os.path.join(root, file))
    return videos


shooting_videos = get_video_paths(VIDEO_DIR)
label1 = os.path.join(DATASET_DIR, "UCFCrime_Test.json")
label2 = os.path.join(DATASET_DIR, "UCFCrime_Train.json")
label3 = os.path.join(DATASET_DIR, "UCFCrime_Val.json")

with open(label1, "r") as f:
    data1 = json.load(f)

with open(label2, "r") as f:
    data2 = json.load(f)

with open(label3, "r") as f:
    data3 = json.load(f)

data_json1 = {}
data_json2 = {}
data_json3 = {}

for video_name, value in data1.items():
    if "Shooting" in video_name:
        data_json1[video_name] = value

for video_name, value in data2.items():
    if "Shooting" in video_name:
        data_json2[video_name] = value

for video_name, value in data3.items():
    if "Shooting" in video_name:
        data_json3[video_name] = value

merged = {**data_json1, **data_json2, **data_json3}
merged

{'Shooting033_x264': {'duration': 121.05,
  'timestamps': [[0.0, 13.4],
   [13.4, 31.1],
   [31.1, 52.5],
   [52.5, 71.6],
   [71.6, 79.5],
   [79.5, 88.0],
   [88.0, 112.3],
   [112.3, 119.2]],
  'sentences': ['A white off-road vehicle reversed, and a black car drove behind the white off-road vehicle.',
   'A black car drove past quickly and then drove away.',
   'A man stood at the door, a woman got out of the car, and two men riding models rode over quickly.',
   "A man got off the motorcycle and grabbed the woman's things, then dragged the woman down and ran away.",
   'The man looked at the woman and then ran out. The woman was lying on the ground.',
   'A white car drove across the road and drove away.',
   'A woman in white clothes is running next to the woman, and a man in white clothes is standing next to the woman',
   'A man in white clothes goes to a woman in white clothes lying on the ground.']},
 'Shooting034_x264': {'duration': 47.06,
  'timestamps': [[0.0, 7.3],
   [7.3

In [11]:
weaponize_data = {}
for video_name, data in merged.items():
    video_path = os.path.join(VIDEO_DIR, f"{video_name}.mp4")
    timestamps = data["timestamps"]
    sentences = data["sentences"]
    weaponized_indices = []
    print(video_name)
    for i, sentence in enumerate(sentences):
        if (
            "firearm" in sentence
            or "weapon" in sentence
            or "gun" in sentence
            or "pistol" in sentence
            or "shoot" in sentence
            or "revolver" in sentence
            or "rifle" in sentence
        ):
            weaponized_indices.append(i)
        else:
            print(sentence)

    weaponized_timestamps = [timestamps[i] for i in weaponized_indices]
    weaponize_data[video_name] = weaponized_timestamps

Shooting033_x264
A white off-road vehicle reversed, and a black car drove behind the white off-road vehicle.
A black car drove past quickly and then drove away.
A man stood at the door, a woman got out of the car, and two men riding models rode over quickly.
A man got off the motorcycle and grabbed the woman's things, then dragged the woman down and ran away.
The man looked at the woman and then ran out. The woman was lying on the ground.
A white car drove across the road and drove away.
A woman in white clothes is running next to the woman, and a man in white clothes is standing next to the woman
A man in white clothes goes to a woman in white clothes lying on the ground.
Shooting034_x264
A red SUV is driving on the road
The red SUV stopped in front of the zebra crossing
A black car is driving on the road
The black car is parked on the road
The door of the black car opened, and a person walked out onto the road
The red SUV is moving forward
People on the road turned around and ran to 

In [12]:
weaponize_data

{'Shooting033_x264': [],
 'Shooting034_x264': [[31.3, 35.6]],
 'Shooting036_x264': [[5.2, 7.2]],
 'Shooting037_x264': [],
 'Shooting038_x264': [[12.1, 21.2], [29.3, 34.6], [38.2, 43.0]],
 'Shooting039_x264': [[9.7, 18.6], [22.6, 26.1]],
 'Shooting040_x264': [[15.1, 18.7], [17.6, 20.4]],
 'Shooting041_x264': [],
 'Shooting042_x264': [[34.1, 41.7]],
 'Shooting043_x264': [[24.6, 36.7], [36.7, 62.5]],
 'Shooting044_x264': [],
 'Shooting046_x264': [[134.4, 145.6], [158.6, 168.2]],
 'Shooting001_x264': [[1.1, 5.2], [5.1, 6.2], [6.1, 8.1]],
 'Shooting002_x264': [],
 'Shooting003_x264': [],
 'Shooting004_x264': [[11.4, 14.4], [14.9, 18.9]],
 'Shooting005_x264': [[52.3, 60.3]],
 'Shooting006_x264': [[91.9, 101.9], [125.0, 135.0]],
 'Shooting007_x264': [],
 'Shooting008_x264': [[8.3, 12.3]],
 'Shooting009_x264': [],
 'Shooting010_x264': [[40.2, 50.2], [45.2, 48.2]],
 'Shooting011_x264': [[26.9, 37.4]],
 'Shooting012_x264': [[27.6, 37.6]],
 'Shooting013_x264': [[29.9, 30.9]],
 'Shooting014_x264':

In [17]:
import cv2

OUTPUT_DIR = "/Volumes/TeeExt/violence"
START_INDEX = 0  # Use for test only


def cut_video_clips(
    video_path: str,
    timestamps: list,
    output_prefix="ucf_crime",
) -> None:

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter.fourcc(*"mp4v")

    global START_INDEX
    for start_sec, end_sec in timestamps:
        start_frame = int(start_sec * fps)
        end_frame = int(end_sec * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        out = cv2.VideoWriter(
            os.path.join(
                OUTPUT_DIR, f"{output_prefix}_video_{START_INDEX + 1}.mp4"
            ),
            fourcc,
            fps,
            (width, height),
        )
        for frame_idx in range(start_frame, end_frame):
            ret, frame = cap.read()
            if not ret:
                break
            out.write(frame)
        out.release()
        START_INDEX += 1
    cap.release()


for video_name, ts_list in weaponize_data.items():
    video_path = os.path.join(VIDEO_DIR, f"{video_name}.mp4")
    if not ts_list:
        continue

    cut_video_clips(video_path, ts_list)
